In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import pearsonr

# --- CONFIG ---
EMBEDDING_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data")
CLASSIFY_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Classify_Data")
CLASSIFY_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
f = r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data\embeddings_openai_1678_1700.parquet"
df = pd.read_parquet(f)

In [3]:
df

,original_index,labels,embedding
0,91,2,"[0.0585700124502182, -0.053055573254823685, -0..."
1,13,2,"[0.008811351843178272, -0.05566640570759773, -..."
2,4,2,"[0.02825016714632511, -0.030990414321422577, 0..."
3,23,2,"[0.024680456146597862, -0.05186709389090538, 0..."
4,74,2,"[0.004600805696099997, -0.07876764237880707, -..."
...,...,...,...
4391,27,2,"[0.04727917164564133, -0.047429025173187256, -..."
4392,69,2,"[0.06103033944964409, -0.0262727253139019, 0.0..."
4393,18,2,"[0.03716849163174629, -0.0483926422894001, -0...."
4394,53,2,"[0.0027602219488471746, -0.05661119148135185, ..."


<h1> Lasso </h1>

In [4]:
def train_lasso_per_period():
    # Find all embedding files
    files = list(EMBEDDING_DIR.glob("embeddings_openai_*.parquet"))
    
    if not files:
        print("No embedding files found. Please run Step 3.a first.")
        return

    results_summary = []

    for f in files:
        # Extract period name (e.g., 1701_1725) for file naming
        period_name = f.name.replace("embeddings_", "").replace(".parquet", "")
        print(f"\n--- Processing Period: {period_name} ---")
        
        # 1. Load Data
        df = pd.read_parquet(f)

        
        # Expand X (embeddings) and y (labels)
        X = np.stack(df['embedding'].values)
        y = df['labels'].values

        # 2. Split (OOS Evaluation)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        pipeline = make_pipeline(
            StandardScaler(), 
            LassoCV(cv=5, random_state=42, n_jobs=-1)
        )

        # 2. Fit the pipeline
        pipeline.fit(X_train, y_train)

        # 3. Access the fitted model from the pipeline to get stats
        lasso_model = pipeline.named_steps['lassocv']
        
        # 4. Evaluate
        preds = pipeline.predict(X_test)
        # preds_clipped = np.clip(preds, 1, 5)
        preds_clipped = preds 


        mse = mean_squared_error(y_test, preds_clipped)
        r2 = r2_score(y_test, preds_clipped)
        corr, _ = pearsonr(y_test, preds_clipped)

        # 5. Save individual Period Results
        period_stats = {
            "period": period_name,
            "oos_mse": mse,
            "oos_r2": r2,
            "oos_pearson": corr,
            "best_alpha": lasso_model.alpha_,
            "nonzero_coeffs": np.count_nonzero(lasso_model.coef_)
        }
        
        # Save to CSV in Classify_Data
        output_path = CLASSIFY_DIR / f"lasso_performance_{period_name}.csv"
        pd.DataFrame([period_stats]).to_csv(output_path, index=False)
        print(f"Results saved to: {output_path}")
        
        results_summary.append(period_stats)

    # Final Summary Table for all periods combined
    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv(CLASSIFY_DIR / "overall_lasso_summary.csv", index=False)
    print("\nAll periods processed. Summary saved to overall_lasso_summary.csv")

In [5]:
if __name__ == "__main__":
    train_lasso_per_period()


--- Processing Period: openai_1678_1700 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.184e-01, tolerance: 6.417e-02
  model = cd_fast.enet_coordinate_descent(
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.706e-02, tolerance: 6.326e-02
  model = cd_fast.enet_coordinate_descent(
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider in

KeyboardInterrupt: 

<h1> XGBoost </h1>

In [27]:
import pandas as pd
import numpy as np
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

In [28]:
def train_xgb_per_period():
    embed_type = "bge"
    files = list(EMBEDDING_DIR.glob(f"embeddings_{embed_type}_*.parquet"))
    
    if not files:
        print("No embedding files found.")
        return

    results_summary = []

    for f in files:
        period_name = f.name.replace(f"embeddings_{embed_type}_", "").replace(".parquet", "")
        print(f"\n--- Training Ridge for: {period_name} ---")
        
        df = pd.read_parquet(f)
        X = np.stack(df['embedding'].values)
        y = df['labels'].values

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 1. Initialize XGBRegressor
        # We use standard hyper-parameters for an initial baseline
        xgb_model = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=-1,
            random_state=42,
            objective='reg:squarederror'
        )

        # 2. Fit with Early Stopping to prevent over-fitting
        xgb_model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )

        # 3. Evaluate
        preds = xgb_model.predict(X_test)
        
        mse = mean_squared_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        corr, _ = pearsonr(y_test, preds)

        period_stats = {
            "period": period_name,
            "model": "XGBoost",
            "oos_mse": mse,
            "oos_r2": r2,
            "oos_pearson": corr
        }
        
        # Save individual report
        output_path = CLASSIFY_DIR / f"xgb_performance_{period_name}_{embed_type}.csv"
        pd.DataFrame([period_stats]).to_csv(output_path, index=False)
        print(f"XGBoost R2 for {period_name}: {r2:.4f}")
        
        results_summary.append(period_stats)

    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv(CLASSIFY_DIR / f"overall_xgb_summary_{embed_type}.csv.csv", index=False)
    print("\nXGBoost processing complete.")

In [29]:
if __name__ == "__main__":
    train_xgb_per_period()


--- Training Ridge for: 1678_1700 ---
XGBoost R2 for 1678_1700: 0.2172

--- Training Ridge for: 1701_1725 ---
XGBoost R2 for 1701_1725: 0.3205

--- Training Ridge for: 1701_1750 ---
XGBoost R2 for 1701_1750: 0.3198

--- Training Ridge for: 1726_1750 ---
XGBoost R2 for 1726_1750: 0.2770

--- Training Ridge for: 1751_1775 ---
XGBoost R2 for 1751_1775: 0.2845

--- Training Ridge for: 1751_1800 ---
XGBoost R2 for 1751_1800: 0.3019

--- Training Ridge for: 1776_1800 ---
XGBoost R2 for 1776_1800: 0.2903

--- Training Ridge for: 1801_1825 ---
XGBoost R2 for 1801_1825: 0.3235

--- Training Ridge for: 1801_1850 ---
XGBoost R2 for 1801_1850: 0.3568

--- Training Ridge for: 1826_1850 ---
XGBoost R2 for 1826_1850: 0.2812

--- Training Ridge for: 1876_1900 ---


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

# --- CONFIG ---
EMBEDDING_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data")
CLASSIFY_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Classify_Data")
CLASSIFY_DIR.mkdir(parents=True, exist_ok=True)

def train_xgb_per_period():
    embed_type = "bge"
    files = list(EMBEDDING_DIR.glob(f"embeddings_{embed_type}_*.parquet"))
    
    if not files:
        print("No embedding files found.")
        return

    results_summary = []

    for f in files:
        period_name = f.name.replace(f"embeddings_{embed_type}_", "").replace(".parquet", "")
        print(f"\n--- Training XGBoost for: {period_name} ---")
        
        df = pd.read_parquet(f)
        X = np.stack(df['embedding'].values)
        y = df['labels'].values

        # 1. Split into 10k-style Training (80%) and Evaluation (20%)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 2. STANDARDIZATION: Fit ONLY on Training data
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train) # Learns mean/std from 10k training
        X_test_scaled = scaler.transform(X_test)      # Applies that exact ruler to test set

        # 3. Initialize XGBRegressor
        xgb_model = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=-1,
            random_state=42,
            objective='reg:squarederror'
        )

        # 4. Fit with Early Stopping on the SCALED evaluation set
        xgb_model.fit(
            X_train_scaled, y_train,
            eval_set=[(X_test_scaled, y_test)],
            verbose=False
        )

        # 5. Evaluate
        preds = xgb_model.predict(X_test_scaled)
        # Use unclipped for R2/MSE reporting unless professor specifically requested clipped
        
        mse = mean_squared_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        corr, _ = pearsonr(y_test, preds)

        # 6. Save Model and Scaler for the "Full Cycle" (Inference)
        joblib.dump(scaler, CLASSIFY_DIR / f"scaler_xgb_{period_name}_{embed_type}.pkl")
        joblib.dump(xgb_model, CLASSIFY_DIR / f"xgb_model_{period_name}_{embed_type}.pkl")

        period_stats = {
            "period": period_name,
            "model": "XGBoost",
            "oos_mse": mse,
            "oos_r2": r2,
            "oos_pearson": corr
        }
        
        output_path = CLASSIFY_DIR / f"xgb_performance_{period_name}_{embed_type}.csv"
        pd.DataFrame([period_stats]).to_csv(output_path, index=False)
        print(f"XGBoost R2 for {period_name}: {r2:.4f}")
        
        results_summary.append(period_stats)

    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv(CLASSIFY_DIR / f"overall_xgb_summary_{embed_type}.csv", index=False)
    print("\nXGBoost processing complete. Scalers and Models saved.")

if __name__ == "__main__":
    train_xgb_per_period()

<h1> Ridge </h1>

In [45]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
from pathlib import WindowsPath
import matplotlib.pyplot as plt
import seaborn as sns


# --- CONFIG ---
EMBEDDING_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data")
CLASSIFY_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Classify_Data")
CLASSIFY_DIR.mkdir(parents=True, exist_ok=True)

In [46]:
def train_ridge_per_period():
    embed_type = "bge"
    files = list(EMBEDDING_DIR.glob(f"embeddings_{embed_type}_*.parquet"))

    if not files:
        print("No OpenAI embedding files found.")
        return

    results_summary = []

    for f in files:
        period_name = f.name.replace(f"embeddings_{embed_type}_", "").replace(".parquet", "")
        print(f"\n--- Training Ridge for: {period_name} ---")
        
        df = pd.read_parquet(f)
        X = np.stack(df['embedding'].values)
        y = df['labels'].values

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 1. Create a Pipeline: Scale -> Ridge with Cross-Validation
        # RidgeCV automatically finds the best 'alpha' (lambda)
        pipeline = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=np.logspace(-6, 6, 50), cv=10)
        )

        # 2. Fit the model
        pipeline.fit(X_train, y_train)
        ridge_model = pipeline.named_steps['ridgecv']

        # 3. Evaluate
        preds = pipeline.predict(X_test)
        # preds_clipped = np.clip(preds, 1, 5)
        preds_clipped = preds

        mse = mean_squared_error(y_test, preds_clipped)
        r2 = r2_score(y_test, preds_clipped)
        corr, _ = pearsonr(y_test, preds_clipped)

        period_stats = {
            "period": period_name,
            "model": "Ridge",
            "oos_mse": mse,
            "oos_r2": r2,
            "oos_pearson": corr,
            "best_alpha": ridge_model.alpha_
        }
        
        # Save individual report
        output_path = CLASSIFY_DIR / f"ridge_performance_{period_name}_{embed_type}.csv"
        pd.DataFrame([period_stats]).to_csv(output_path, index=False)
        print(f"Ridge R2 for {period_name}: {r2:.4f} (Alpha: {ridge_model.alpha_})")
        
        results_summary.append(period_stats)

        # --- NEW: THRESHOLD SENSITIVITY ANALYSIS ---
        # We focus on the "True High Quality" group (Actual labels 4 or 5)
        high_quality_mask = (y_test >= 4)
        y_test_high = y_test[high_quality_mask]
        preds_high = preds[high_quality_mask]
        
        thresholds = [2.0, 2.5, 3.0, 3.5, 4.0]
        error_results = []
        
        print(f"\nQuality Retention Audit for {period_name}:")
        for t in thresholds:
            # How many of the 4/5s are predicted BELOW the threshold?
            misclassified = (preds_high < t).sum()
            error_rate = misclassified / len(y_test_high)
            
            error_results.append({
                "threshold": t,
                "misclassified_count": misclassified,
                "error_rate": error_rate
            })
            print(f"  Threshold {t}: Losing {error_rate:.1%} of high-quality data ({misclassified}/{len(y_test_high)})")

        # Optional: Save this audit to a CSV
        audit_df = pd.DataFrame(error_results)
        audit_df.to_csv(CLASSIFY_DIR / f"quality_audit_{period_name}_{embed_type}.csv", index=False)

        # Quick code snippet to add inside the loop
        plot_df = pd.DataFrame({'Actual': y_test, 'Predicted': preds})
        plt.figure(figsize=(10, 6))
        sns.boxplot(x='Actual', y='Predicted', data=plot_df)
        plt.axhline(y=3.0, color='r', linestyle='--', label='Threshold 3.0')
        plt.title(f"Prediction Distribution for {period_name}")
        plt.savefig(CLASSIFY_DIR / f"box_plot_{period_name}.png")
        plt.close()

    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv(CLASSIFY_DIR / f"overall_ridge_summary_{embed_type}.csv", index=False)
    print("\nRidge processing complete.")



In [47]:
if __name__ == "__main__":
    train_ridge_per_period()


--- Training Ridge for: 1678_1700 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=3.79767e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.21141e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=2.77146e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=3.27574e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Ridge R2 for 1678_1700: 0.2765 (Alpha: 2023.5896477251556)

Quality Retention Audit for 1678_1700:
  Threshold 2.0: Losing 0.0% of high-quality data (0/4)
  Threshold 2.5: Losing 100.0% of high-quality data (4/4)
  Threshold 3.0: Losing 100.0% of high-quality data (4/4)
  Threshold 3.5: Losing 100.0% of high-quality data (4/4)
  Threshold 4.0: Losing 100.0% of high-quality data (4/4)

--- Training Ridge for: 1701_1750 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.26472e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.64737e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.95522e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.61714e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Ridge R2 for 1701_1750: 0.3855 (Alpha: 212.09508879201925)

Quality Retention Audit for 1701_1750:
  Threshold 2.0: Losing 5.3% of high-quality data (1/19)
  Threshold 2.5: Losing 84.2% of high-quality data (16/19)
  Threshold 3.0: Losing 100.0% of high-quality data (19/19)
  Threshold 3.5: Losing 100.0% of high-quality data (19/19)
  Threshold 4.0: Losing 100.0% of high-quality data (19/19)

--- Training Ridge for: 1751_1800 ---


...

Ridge R2 for 1901_1950: 0.5240 (Alpha: 120.67926406393264)

Quality Retention Audit for 1901_1950:
  Threshold 2.0: Losing 0.1% of high-quality data (1/985)
  Threshold 2.5: Losing 1.9% of high-quality data (19/985)
  Threshold 3.0: Losing 19.3% of high-quality data (190/985)
  Threshold 3.5: Losing 48.3% of high-quality data (476/985)
  Threshold 4.0: Losing 90.9% of high-quality data (895/985)

--- Training Ridge for: 1951_2000 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.96274e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.25271e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.63265e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.34585e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Ridge R2 for 1951_2000: 0.6331 (Alpha: 120.67926406393264)

Quality Retention Audit for 1951_2000:
  Threshold 2.0: Losing 0.0% of high-quality data (0/2617)
  Threshold 2.5: Losing 0.2% of high-quality data (6/2617)
  Threshold 3.0: Losing 2.6% of high-quality data (68/2617)
  Threshold 3.5: Losing 11.5% of high-quality data (301/2617)
  Threshold 4.0: Losing 43.4% of high-quality data (1136/2617)

--- Training Ridge for: 2001_2023 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=9.16779e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.03152e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=9.73009e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.01248e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Ridge R2 for 2001_2023: 0.5937 (Alpha: 372.7593720314938)

Quality Retention Audit for 2001_2023:
  Threshold 2.0: Losing 0.0% of high-quality data (0/1190)
  Threshold 2.5: Losing 0.0% of high-quality data (0/1190)
  Threshold 3.0: Losing 0.9% of high-quality data (11/1190)
  Threshold 3.5: Losing 8.5% of high-quality data (101/1190)
  Threshold 4.0: Losing 38.0% of high-quality data (452/1190)

Ridge processing complete.


In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
from sklearn.utils import resample

# --- CONFIG ---
EMBEDDING_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data")
CLASSIFY_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Classify_Data")
CLASSIFY_DIR.mkdir(parents=True, exist_ok=True)

def train_ridge_per_period():
    embed_type = "bge"
    files = list(EMBEDDING_DIR.glob(f"embeddings_{embed_type}_*.parquet"))
    
    if not files:
        print("No embedding files found.")
        return

    results_summary = []

    for f in files:
        period_name = f.name.replace(f"embeddings_{embed_type}_", "").replace(".parquet", "")
        print(f"\n--- Processing Period: {period_name} ---")
        
        df = pd.read_parquet(f)
        
        # 1. Class Distribution Audit
        counts = df['labels'].value_counts().sort_index()
        print("Label Distribution (Initial):")
        for label, count in counts.items():
            print(f"  Score {label}: {count} samples")

        X = np.stack(df['embedding'].values)
        y = df['labels'].values

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 2. Optional: Simple Oversampling for 4s and 5s
        # This helps the model 'weight' the high-quality samples more heavily.
        train_df = pd.DataFrame({'labels': y_train})
        train_df['X_idx'] = range(len(y_train))
        
        high_quality = train_df[train_df['labels'] >= 4]
        if not high_quality.empty and len(high_quality) < len(train_df) * 0.1:
            print(f"  Oversampling {len(high_quality)} high-quality samples to balance training...")
            oversampled = resample(high_quality, replace=True, n_samples=int(len(y_train)*0.2), random_state=42)
            # Add oversampled indices back to training set
            new_indices = np.concatenate([train_df['X_idx'].values, oversampled['X_idx'].values])
            X_train = X_train[new_indices]
            y_train = y[new_indices]

        # 3. Create a Pipeline
        pipeline = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=np.logspace(-3, 6, 50), cv=10)
        )

        # 4. Fit and Predict
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)

        # 5. Visual QC: Box Plot
        plot_df = pd.DataFrame({'Actual': y_test, 'Predicted': preds})
        plt.figure(figsize=(10, 6))
        sns.boxplot(x='Actual', y='Predicted', data=plot_df)
        plt.axhline(y=3.0, color='r', linestyle='--', label='Target Threshold')
        plt.title(f"Improved Prediction Distribution for {period_name}")
        plt.savefig(CLASSIFY_DIR / f"box_plot_{period_name}_{embed_type}.png")
        plt.close()

        # 6. Final Evaluation
        mse = mean_squared_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        
        results_summary.append({
            "period": period_name,
            "oos_r2": r2,
            "best_alpha": pipeline.named_steps['ridgecv'].alpha_
        })

    print("\nProcessing complete. Check box plots for distribution improvements.")

if __name__ == "__main__":
    train_ridge_per_period()


--- Processing Period: 1678_1700 ---
Label Distribution (Initial):
  Score 1: 1246 samples
  Score 2: 3099 samples
  Score 3: 42 samples
  Score 4: 8 samples
  Score 5: 1 samples
  Oversampling 5 high-quality samples to balance training...


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.8991e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.90009e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.28635e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.67632e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C


--- Processing Period: 1701_1725 ---
Label Distribution (Initial):
  Score 1: 3135 samples
  Score 2: 6765 samples
  Score 3: 68 samples
  Score 4: 27 samples
  Score 5: 5 samples
  Oversampling 29 high-quality samples to balance training...


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.57908e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.44682e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.40253e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.37077e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T



--- Processing Period: 1701_1750 ---
Label Distribution (Initial):
  Score 1: 5134 samples
  Score 2: 14521 samples
  Score 3: 258 samples
  Score 4: 82 samples
  Score 5: 5 samples
  Oversampling 68 high-quality samples to balance training...


...


--- Processing Period: 1976_2000 ---
Label Distribution (Initial):
  Score 1: 2 samples
  Score 2: 756 samples
  Score 3: 405 samples
  Score 4: 6185 samples
  Score 5: 2602 samples


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.72833e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.19731e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.81001e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.51934e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T



--- Processing Period: 2001_2023 ---
Label Distribution (Initial):
  Score 1: 9 samples
  Score 2: 996 samples
  Score 3: 1516 samples
  Score 4: 4339 samples
  Score 5: 1575 samples


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.42633e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.47915e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.44628e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.39495e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T



Processing complete. Check box plots for distribution improvements.


<h2> High Quality Label Plot </h2>

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
from sklearn.utils import resample

# --- CONFIG ---
EMBEDDING_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data")
CLASSIFY_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Classify_Data")
CLASSIFY_DIR.mkdir(parents=True, exist_ok=True)

def train_ridge_per_period():
    embed_type = "bge"
    files = list(EMBEDDING_DIR.glob(f"embeddings_{embed_type}_*.parquet"))
    
    if not files:
        print("No embedding files found.")
        return

    results_summary = []

    for f in files:
        period_name = f.name.replace(f"embeddings_{embed_type}_", "").replace(".parquet", "")
        print(f"\n--- Processing Period: {period_name} ---")
        
        df = pd.read_parquet(f)
        X = np.stack(df['embedding'].values)
        y = df['labels'].values

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 1. Oversampling logic (as before)
        train_df = pd.DataFrame({'labels': y_train})
        train_df['X_idx'] = range(len(y_train))
        high_quality_train = train_df[train_df['labels'] >= 4]
        
        if not high_quality_train.empty and len(high_quality_train) < len(train_df) * 0.1:
            oversampled = resample(high_quality_train, replace=True, n_samples=int(len(y_train)*0.2), random_state=42)
            new_indices = np.concatenate([train_df['X_idx'].values, oversampled['X_idx'].values])
            X_train = X_train[new_indices]
            y_train = y[new_indices]

        # 2. Pipeline and Training
        pipeline = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 6, 50), cv=10))
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)

        # 3. SENSITIVITY PLOT LOGIC
        # X-axis: Thresholds from 0 to 5
        # Y-axis: Proportion of (Actual 4s and 5s) correctly labeled as "Good"
        thresholds = np.linspace(0, 5, 50)
        
        # Isolate the "Actual Good" entries (Recall focus)
        actual_good_mask = (y_test >= 4)
        num_actual_good = np.sum(actual_good_mask)
        
        proportions = []
        for t in thresholds:
            if num_actual_good > 0:
                # Correctly labeled: Truly >= 4 AND Predicted >= Threshold
                correctly_found = np.sum((y_test >= 4) & (preds >= t))
                recall = correctly_found / num_actual_good
                proportions.append(recall)
            else:
                proportions.append(0)

        # Generate and save the plot
        plt.figure(figsize=(10, 6))
        plt.plot(thresholds, proportions, marker='o', linestyle='-', color='teal', markersize=4)
        plt.xlabel("Quality Threshold (What we call 'Good')")
        plt.ylabel("Proportion of True 4/5s Correctly Labeled")
        plt.title(f"Recall Curve for High-Quality Data: {period_name}")
        plt.grid(True, alpha=0.3)
        plt.ylim(-0.05, 1.05)
        plt.axvline(x=3.0, color='red', linestyle='--', label='Reference T=3.0')
        plt.legend()
        plt.savefig(CLASSIFY_DIR / f"sensitivity_curve_{period_name}_{embed_type}.png")
        plt.close()

        print(f"  Recall at Threshold 2.0: {proportions[int(len(thresholds)*0.4)]:.1%}")

    print("\nProcessing complete. Check sensitivity curves for data retention stats.")

if __name__ == "__main__":
    train_ridge_per_period()


--- Processing Period: 1678_1700 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.8991e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.90009e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.28635e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.67632e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C

  Recall at Threshold 2.0: 0.0%

--- Processing Period: 1701_1725 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.57908e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.44682e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.40253e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.37077e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


  Recall at Threshold 2.0: 0.0%

--- Processing Period: 1726_1750 ---


...

  Recall at Threshold 2.0: 100.0%

--- Processing Period: 1976_2000 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.72833e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.19731e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.81001e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=6.51934e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


  Recall at Threshold 2.0: 100.0%

--- Processing Period: 2001_2023 ---


C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.42633e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.47915e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.44628e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\danielyoon\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=1.39495e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


  Recall at Threshold 2.0: 100.0%

Processing complete. Check sensitivity curves for data retention stats.


<h1> Merge sometime periods </h1>

In [38]:
from pathlib import WindowsPath

merge_list = [
    ("1701", "1725", "1726", "1750"),
    ("1751", "1775", "1776", "1800"),
    ("1801", "1825", "1826", "1850"),
    ("1851", "1875", "1876", "1900"),
    ("1901", "1925", "1926", "1950"),
    ("1951", "1975", "1976", "2000"),
]
for (n1, n2, n3, n4) in merge_list:
    df1 = pd.read_parquet(fr"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data\embeddings_bge_{n1}_{n2}.parquet")
    df2 = pd.read_parquet(fr"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data\embeddings_bge_{n3}_{n4}.parquet")
    df3 = pd.concat([df1, df2])
    df3.to_parquet(fr"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Embedding_Data\embeddings_bge_{n1}_{n4}.parquet")